# Drift Report — per release

Renders the output of `src/drift/measure_drift.py` (the per-class drift metrics) and `src/drift/find_gaps.py` (top gap clusters) into tables + bar charts. Links over to the existing UMAP notebooks for supporting visual evidence.

Inputs:
- `data/processed/drift_report.json` (stage 4)
- `data/processed/gap_clusters.json`  (stage 5, optional — skip cell if missing)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROCESSED = Path('/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed')
DRIFT_FILE = PROCESSED / 'drift_report.json'
GAPS_FILE  = PROCESSED / 'gap_clusters.json'

drift = json.loads(DRIFT_FILE.read_text())
print('embedding types:', [k for k in drift if k != 'meta'])
print('drift meta:', drift['meta'])

## 1. Per-class drift table

In [ ]:
def flatten(report: dict, emb: str) -> pd.DataFrame:
    rows = []
    for cls, cell in report[emb].items():
        if cell.get('skipped'):
            rows.append({'class': cls, 'n_gold': cell.get('n_gold'), 'n_tracing': cell.get('n_tracing'), 'skipped': cell['skipped']})
            continue
        rows.append({
            'embedding': emb,
            'class': cls,
            'n_gold': cell['n_gold'],
            'n_tracing': cell['n_tracing'],
            'centroid_cos': cell['centroid']['cosine_distance'],
            'centroid_l2':  cell['centroid']['l2'],
            'cos_mean':     cell['cosine']['mean'],
            'cos_p25':      cell['cosine']['p25'],
            'nn_jaccard@k': cell['nn']['mean_jaccard_at_k'],
            'js_div':       cell['cluster']['js_divergence'],
            'wasserstein':  cell['wasserstein']['mean'],
            'trace_only_clusters': cell['cluster']['tracing_only_clusters'],
        })
    return pd.DataFrame(rows)

df_openai = flatten(drift, 'openai').sort_values('js_div', ascending=False)
df_tfidf  = flatten(drift, 'tfidf').sort_values('js_div', ascending=False)
df_openai

In [ ]:
df_tfidf

## 2. Drift bar charts — JS divergence + NN Jaccard

**Higher JS = more divergent.** **Lower NN Jaccard = tracing queries have weaker analogues in the gold set.** Together they identify classes most in need of enrichment.

In [ ]:
def drift_bar(df_o: pd.DataFrame, df_t: pd.DataFrame, metric: str, title: str, ascending: bool = False) -> go.Figure:
    rows = []
    for label, df in [('openai', df_o), ('tfidf', df_t)]:
        for _, r in df.iterrows():
            if metric not in r or pd.isna(r[metric]) or r['class'] == '__all__':
                continue
            rows.append({'class': r['class'], 'embedding': label, metric: r[metric]})
    plot_df = pd.DataFrame(rows)
    if plot_df.empty:
        return go.Figure().update_layout(title=f'{title} (no data)')
    order = (plot_df.groupby('class')[metric].mean()
             .sort_values(ascending=ascending).index.tolist())
    fig = px.bar(plot_df, x='class', y=metric, color='embedding', barmode='group',
                 category_orders={'class': order}, title=title)
    fig.update_layout(width=1100, height=480, xaxis_tickangle=-30)
    return fig

drift_bar(df_openai, df_tfidf, 'js_div',       'JS divergence per class (higher = more drift)').show()
drift_bar(df_openai, df_tfidf, 'nn_jaccard@k', 'Nearest-neighbor Jaccard@k (lower = less gold coverage)', ascending=True).show()
drift_bar(df_openai, df_tfidf, 'centroid_cos', 'Centroid cosine distance per class').show()

## 3. Class size mismatch — gold vs tracing

In [ ]:
sizes = df_openai[df_openai['class'] != '__all__'][['class', 'n_gold', 'n_tracing']].copy()
sizes['ratio_t_over_g'] = sizes['n_tracing'] / sizes['n_gold'].clip(lower=1)
sizes = sizes.sort_values('ratio_t_over_g', ascending=False)

fig = go.Figure()
fig.add_bar(name='gold', x=sizes['class'], y=sizes['n_gold'], marker_color='#2979ff')
fig.add_bar(name='tracing', x=sizes['class'], y=sizes['n_tracing'], marker_color='#ff6b6b')
fig.update_layout(barmode='group', title='Per-class record count: gold vs tracing',
                  width=1100, height=460, xaxis_tickangle=-30)
fig.show()
sizes

## 4. Top gap clusters (stage 5)

If `gap_clusters.json` doesn't exist yet, run:
```
python3 src/drift/find_gaps.py
```

In [ ]:
if GAPS_FILE.exists():
    gaps = json.loads(GAPS_FILE.read_text())
    top = pd.DataFrame(gaps['clusters_top']).drop(columns=['sample_tracing_queries'], errors='ignore')
    print(f"meta: {gaps['meta']}")
    display(top)
else:
    print('gap_clusters.json not found — run src/drift/find_gaps.py first')

In [ ]:
if GAPS_FILE.exists():
    rows = []
    for c in gaps['clusters_top']:
        for q in c['sample_tracing_queries'][:3]:
            rows.append({
                'cluster_id': c['cluster_id'],
                'gap_score':  c['gap_score'],
                'top_class':  c['top_classes'][0]['class'] if c['top_classes'] else '?',
                'sample_query': q,
            })
    pd.set_option('display.max_colwidth', 200)
    pd.DataFrame(rows)

## 5. Supporting UMAP evidence

Pre-baked UMAP visualizations live in the existing notebooks:

- `src/umap_embeddings_3d.ipynb` — TF-IDF, single corpus.
- `src/umap_embeddings_3d_openai_vs_tfidf.ipynb` — OpenAI vs TF-IDF, side-by-side.
- `src/umap_embeddings_3d_supervised_subclass.ipynb` — subclass labels, supervised vs unsupervised.

Cross-check: a class with high JS divergence here should appear visibly separated in those plots when colored by `category`.